# 스트레스 점수 예측 V17

**V14 ExtraTrees + Native Categorical CatBoost Ensemble**

- Public 0.1278086의 V14 Blend95를 기준 모델로 유지
- CatBoost는 범주형 변수를 One-Hot Encoding 없이 처리
- 두 가지 5-Fold 분할에서 CatBoost 혼합 비율을 검증
- 두 분할 모두 V14보다 좋은 비율만 선택
- 전처리·파라미터 선택에 Test 데이터 통계를 사용하지 않음

In [1]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [2]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
MODEL_SEEDS = (42, 77, 2026)
CV_SEEDS = (42, 2026)
N_ESTIMATORS = 1200

BASE_QUANTILE = 0.510
POOLED_QUANTILE = 0.505
AGE_QUANTILE = 0.500
CAT_WEIGHTS = np.round(np.arange(0.00, 0.41, 0.05), 2)

WEIGHTED_FEATURES = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio",
    "bone_density",
]
AGE_WEIGHTED_FEATURES = WEIGHTED_FEATURES + [
    "mean_arterial_pressure", "diastolic_blood_pressure",
]

## 2. 데이터 검증과 파생변수

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")


def add_base_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_weighted_features(frame):
    result = add_base_features(frame)
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


def add_age_features(frame):
    result = add_base_features(frame)
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_glucose"] = result["age"] * result["glucose"]
    result["age_cholesterol"] = result["age"] * result["cholesterol"]
    result["age_bone_density"] = result["age"] * result["bone_density"]
    result["age_mean_arterial_pressure"] = (
        result["age"] * result["mean_arterial_pressure"]
    )
    for feature in AGE_WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
    return result


def prepare_catboost(frame):
    result = add_base_features(frame)
    categorical = result.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    for column in categorical:
        result[column] = result[column].fillna("__MISSING__").astype(str)
    return result, categorical


y = train[TARGET].reset_index(drop=True)
raw_train = train.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
raw_test = test.drop(columns=[ID_COLUMN]).reset_index(drop=True)
X_weighted = add_weighted_features(raw_train)
X_test_weighted = add_weighted_features(raw_test)
X_age = add_age_features(raw_train)
X_test_age = add_age_features(raw_test)
X_cat, CAT_COLUMNS = prepare_catboost(raw_train)
X_test_cat, _ = prepare_catboost(raw_test)
print("Train:", train.shape, "| categorical:", CAT_COLUMNS)

Train: (3000, 18) | categorical: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']


## 3. ExtraTrees 전처리

In [4]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_matrix(model, matrix):
    return np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )


def q(matrix, quantile):
    return np.clip(
        np.quantile(matrix, quantile, axis=1),
        0.0,
        1.0,
    )

## 4. Repeated 5-Fold OOF 예측

In [5]:
split_oof = {}
best_iterations = []

for cv_seed in CV_SEEDS:
    cv = KFold(n_splits=5, shuffle=True, random_state=cv_seed)
    oof_v14 = np.zeros(len(y))
    oof_cat = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_weighted), start=1):
        weighted_preprocessor = make_preprocessor(
            X_weighted.iloc[train_idx]
        )
        weighted_train = weighted_preprocessor.fit_transform(
            X_weighted.iloc[train_idx]
        )
        weighted_valid = weighted_preprocessor.transform(
            X_weighted.iloc[valid_idx]
        )

        matrices = {}
        for model_seed in MODEL_SEEDS:
            model = ExtraTreesRegressor(
                n_estimators=N_ESTIMATORS,
                min_samples_leaf=1,
                max_features=1,
                random_state=model_seed,
                n_jobs=-1,
            )
            model.fit(weighted_train, y.iloc[train_idx])
            matrices[model_seed] = tree_matrix(model, weighted_valid)

        pred_base = q(matrices[42], BASE_QUANTILE)
        pred_pooled = q(
            np.concatenate(
                [matrices[seed] for seed in MODEL_SEEDS],
                axis=1,
            ),
            POOLED_QUANTILE,
        )

        age_preprocessor = make_preprocessor(X_age.iloc[train_idx])
        age_train = age_preprocessor.fit_transform(X_age.iloc[train_idx])
        age_valid = age_preprocessor.transform(X_age.iloc[valid_idx])
        age_model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=1,
            max_features=1,
            random_state=42,
            n_jobs=-1,
        )
        age_model.fit(age_train, y.iloc[train_idx])
        pred_age = q(tree_matrix(age_model, age_valid), AGE_QUANTILE)
        pred_aux = np.clip(
            0.70 * pred_pooled + 0.30 * pred_age,
            0.0,
            1.0,
        )
        oof_v14[valid_idx] = np.clip(
            0.95 * pred_base + 0.05 * pred_aux,
            0.0,
            1.0,
        )

        cat_model = CatBoostRegressor(
            loss_function="MAE",
            eval_metric="MAE",
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5.0,
            random_seed=42 + fold,
            allow_writing_files=False,
            verbose=False,
        )
        cat_model.fit(
            X_cat.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=CAT_COLUMNS,
            eval_set=(X_cat.iloc[valid_idx], y.iloc[valid_idx]),
            early_stopping_rounds=150,
            verbose=False,
        )
        oof_cat[valid_idx] = np.clip(
            cat_model.predict(X_cat.iloc[valid_idx]),
            0.0,
            1.0,
        )
        best_iterations.append(cat_model.get_best_iteration() + 1)
        print(f"CV seed {cv_seed} | Fold {fold}: complete")

    split_oof[cv_seed] = {"v14": oof_v14, "cat": oof_cat}
    print(
        f"CV seed {cv_seed} | V14={mean_absolute_error(y, oof_v14):.6f} | "
        f"CatBoost={mean_absolute_error(y, oof_cat):.6f}"
    )

CV seed 42 | Fold 1: complete
CV seed 42 | Fold 2: complete
CV seed 42 | Fold 3: complete
CV seed 42 | Fold 4: complete
CV seed 42 | Fold 5: complete
CV seed 42 | V14=0.149080 | CatBoost=0.215308
CV seed 2026 | Fold 1: complete
CV seed 2026 | Fold 2: complete
CV seed 2026 | Fold 3: complete
CV seed 2026 | Fold 4: complete
CV seed 2026 | Fold 5: complete
CV seed 2026 | V14=0.145586 | CatBoost=0.216089


## 5. 안정적인 블렌딩 비율 선택

In [6]:
rows = []
for cat_weight in CAT_WEIGHTS:
    split_scores = []
    improvements = []
    for cv_seed in CV_SEEDS:
        oof_v14 = split_oof[cv_seed]["v14"]
        oof_cat = split_oof[cv_seed]["cat"]
        prediction = np.clip(
            (1.0 - cat_weight) * oof_v14
            + cat_weight * oof_cat,
            0.0,
            1.0,
        )
        base_mae = mean_absolute_error(y, oof_v14)
        score = mean_absolute_error(y, prediction)
        split_scores.append(score)
        improvements.append(base_mae - score)
    rows.append(
        {
            "cat_weight": cat_weight,
            "v14_weight": 1.0 - cat_weight,
            "mean_CV_MAE": np.mean(split_scores),
            "mean_improvement": np.mean(improvements),
            "worst_improvement": min(improvements),
            "wins": sum(value > 0 for value in improvements),
        }
    )

results = pd.DataFrame(rows).sort_values(
    ["wins", "worst_improvement", "mean_CV_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)
eligible = results[
    (results["wins"] == len(CV_SEEDS))
    & (results["worst_improvement"] > 0)
]
if len(eligible):
    selected = eligible.iloc[0]
else:
    selected = results[results["cat_weight"] == 0].iloc[0]

SELECTED_CAT_WEIGHT = float(selected["cat_weight"])
SELECTED_V14_WEIGHT = 1.0 - SELECTED_CAT_WEIGHT
FINAL_CAT_ITERATIONS = int(np.median(best_iterations))

print("Selected V14 weight:", SELECTED_V14_WEIGHT)
print("Selected CatBoost weight:", SELECTED_CAT_WEIGHT)
print("Repeated-CV mean MAE:", f"{selected['mean_CV_MAE']:.6f}")
print("Worst split improvement:", f"{selected['worst_improvement']:+.6f}")
print("Wins across CV splits:", int(selected["wins"]), "/", len(CV_SEEDS))
print("Final CatBoost iterations:", FINAL_CAT_ITERATIONS)
results

Selected V14 weight: 1.0
Selected CatBoost weight: 0.0
Repeated-CV mean MAE: 0.147333
Worst split improvement: +0.000000
Wins across CV splits: 0 / 2
Final CatBoost iterations: 1998


,cat_weight,v14_weight,mean_CV_MAE,mean_improvement,worst_improvement,wins
0,0.00,1.00,0.147333,0.000000,0.000000,0
1,0.05,0.95,0.150521,-0.003188,-0.003292,0
2,0.10,0.90,0.153728,-0.006395,-0.006591,0
3,0.15,0.85,0.156952,-0.009619,-0.009908,0
4,0.20,0.80,0.160189,-0.012856,-0.013237,0
5,0.25,0.75,0.163454,-0.016121,-0.016595,0
6,0.30,0.70,0.166738,-0.019405,-0.019977,0
7,0.35,0.65,0.170053,-0.022721,-0.023381,0
8,0.40,0.60,0.173393,-0.026060,-0.026806,0


## 6. 전체 Train 학습과 Test 예측

In [7]:
weighted_preprocessor = make_preprocessor(X_weighted)
weighted_train = weighted_preprocessor.fit_transform(X_weighted)
weighted_test = weighted_preprocessor.transform(X_test_weighted)

matrices = {}
for model_seed in MODEL_SEEDS:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=model_seed,
        n_jobs=-1,
    )
    model.fit(weighted_train, y)
    matrices[model_seed] = tree_matrix(model, weighted_test)
    print(f"Weighted seed {model_seed}: complete")

pred_base = q(matrices[42], BASE_QUANTILE)
pred_pooled = q(
    np.concatenate([matrices[s] for s in MODEL_SEEDS], axis=1),
    POOLED_QUANTILE,
)

age_preprocessor = make_preprocessor(X_age)
age_train = age_preprocessor.fit_transform(X_age)
age_test = age_preprocessor.transform(X_test_age)
age_model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=1,
    max_features=1,
    random_state=42,
    n_jobs=-1,
)
age_model.fit(age_train, y)
pred_age = q(tree_matrix(age_model, age_test), AGE_QUANTILE)
pred_aux = np.clip(0.70 * pred_pooled + 0.30 * pred_age, 0.0, 1.0)
pred_v14 = np.clip(0.95 * pred_base + 0.05 * pred_aux, 0.0, 1.0)

if SELECTED_CAT_WEIGHT > 0:
    final_cat = CatBoostRegressor(
        loss_function="MAE",
        iterations=FINAL_CAT_ITERATIONS,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        random_seed=42,
        allow_writing_files=False,
        verbose=False,
    )
    final_cat.fit(X_cat, y, cat_features=CAT_COLUMNS, verbose=False)
    pred_cat = np.clip(final_cat.predict(X_test_cat), 0.0, 1.0)
else:
    pred_cat = pred_v14.copy()

pred_v17 = np.clip(
    SELECTED_V14_WEIGHT * pred_v14
    + SELECTED_CAT_WEIGHT * pred_cat,
    0.0,
    1.0,
)
print("Prediction range:", pred_v17.min(), pred_v17.max())
print("첫 10개 예측값:", pred_v17[:10])

Weighted seed 42: complete
Weighted seed 77: complete
Weighted seed 2026: complete
Prediction range: 0.0 1.0
첫 10개 예측값: [0.56055 0.97    0.19    0.44015 0.53    0.46    0.78    0.4697  0.5505
 0.88   ]


## 7. 제출 파일 저장

In [8]:
submission_v17 = submission.copy()
submission_v17[ID_COLUMN] = test[ID_COLUMN]
submission_v17[TARGET] = pred_v17

output_path = OUTPUT_DIR / "submit_v17_v14_catboost.csv"
submission_v17.to_csv(output_path, index=False, encoding="utf-8")
print("Saved:", output_path)
print("Rows:", len(submission_v17))
submission_v17.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v17_v14_catboost.csv
Rows: 3000


,ID,stress_score
0,TEST_0000,0.56055
1,TEST_0001,0.97000
2,TEST_0002,0.19000
3,TEST_0003,0.44015
4,TEST_0004,0.53000
5,TEST_0005,0.46000
6,TEST_0006,0.78000
7,TEST_0007,0.46970
8,TEST_0008,0.55050
9,TEST_0009,0.88000


## 제출 기준

- `Wins across CV splits: 2 / 2`
- `Worst split improvement`가 양수
- 선택된 CatBoost 비중이 0보다 큼 때만 V17을 제출 후보로 검토
- 조건을 만족하지 못하면 현재 1등 V14 Blend95를 유지